# Object Detection

Apply your new knowledge of CNNs to one of the hottest (and most challenging!) fields in computer vision: object detection.

**Learning Objectives**
* Identify the components used for object detection (landmark, anchor, bounding box, grid, ...) and their purpose
* Implement object detection
* Implement non-max suppression to increase accuracy
* Implement intersection over union
* Handle bounding boxes, a type of image annotation popular in deep learning
* Apply sparse categorical crossentropy for pixelwise prediction
* Implement semantic image segmentation on the CARLA self-driving car dataset
* Explain the difference between a regular CNN and a U-net
* Build a U-Net

---

## Table of Contents

---

## Object Localization

This section introduces the transition from simple **Image Classification** to **Object Localization** and eventually **Object Detection**. Here is the summary of the key concepts.

* **Classification:** Identifying *what* is in an image (e.g., "this is a car").
* **Classification with Localization:** Identifying the object *and* drawing a bounding box around it (usually for a single object).
* **Detection:** Identifying and localizing *multiple* objects of different categories within the same image.

<div style='display:flex; justify-content:center'>
    <img src='images/class_local_detection.png' width='750px'>
</div>

### Defining the Bounding Box

To localize an object, the network is trained to output four specific real numbers:

* **($b_x, b_y$):** The coordinates of the center point of the object.
* **$b_h$:** The height of the bounding box (expressed as a fraction of total image height).
* **$b_w$:** The width of the bounding box (expressed as a fraction of total image width).
* **Coordinate System:** The top-left of the image is $(0, 0)$ and the bottom-right is $(1, 1)$.

<div style='display:flex; justify-content:center'>
    <img src='images/localization.png' width='750px'>
</div>

### The Target Label Vector ($y$)

In a supervised learning task for localization, the target label $y$ is structured as a multi-component vector (in this example, an 8-element vector):

$$y = [p_c, b_x, b_y, b_h, b_w, c_1, c_2, c_3]^T$$

* **$p_c$ (Probability of Class):** A binary flag where 1 means an object is present and 0 means only background is present.
* **$b_x, b_y, b_h, b_w$:** The bounding box parameters.
* **$c_1, c_2, c_3$:** The class labels (e.g., Pedestrian, Car, Motorcycle).
* **"Don't Cares":** If $p_c=0$, all other elements in the vector are ignored during training.

### The Loss Function

The loss function is calculated differently depending on whether an object is present in the ground truth:

* **If Object Present ($y_1 = 1$):** The loss is typically the sum of squared errors across all 8 components:

$$L(\hat y, y) = \Sigma_{i=1}^8 (\hat y_i - y_i)^2$$

* **If No Object ($y_1 = 0$):** The loss only considers the error in the $p_c$ prediction:

$$L(\hat y, y) = (\hat y_i - y_i)^2$$

> **Note:** While squared error is used for simplicity, specialized losses like **Log Loss** for classes ($c_i$) and $p_c$, or **IoU-based losses** for bounding boxes, are often used in advanced applications.

---

## Landmark Detection

This section explains **Landmark Detection**, a more granular version of localization where the neural network predicts specific coordinates for points of interest rather than just a general bounding box.

### Key Concepts in Landmark Detection

* **Beyond Bounding Boxes:** Instead of just outputting width and height, the network outputs the specific ($x, y$) coordinates for multiple "landmarks" (key points) that define the shape or pose of an object.
* **Vector Construction:** To detect $N$ landmarks, the network outputs a vector $y$ containing:
    * $P_c$: A confidence score (is an object present?).
    * $2N$ coordinates: $(l_{1x}​,l_{1y}​,l_{2x}​,l_{2y}​,\dots, l_{Nx}​,l_{Ny}​)$.
    * *Example:* For 64 facial landmarks, the model would have 129 output units.
* **Consistency is Critical:** For the model to learn, the "identity" of each landmark must be identical across every image in the dataset (e.g., $l_1$ must *always* represent the left corner of the left eye).

<div style='display:flex; justify-content:center'>
    <img src='images/landmark_detection.png' width='750px'>
</div>

### Common Applications

* **Facial Expression & Emotion Recognition:** By tracking landmarks around the mouth and eyes, models can determine if a person is smiling, frowning, or surprised.
* **Augmented Reality (AR) Filters:** Apps like Snapchat use these landmarks to warp faces or accurately place digital objects (like crowns or hats) on a user's head.
* **Human Pose Estimation:** Detecting the "skeleton" of a person by placing landmarks on joints (shoulders, elbows, wrists, knees) to recognize activities or body language.

### Implementation Summary

| Feature | Bounding Box (Localization) | Landmark Detection |
| --- | --- | --- |
| **Output Type** | 4 Numbers ($(b_x​,b_y​,b_h​,b_w​)$) | 2 $\times$ Numbers (Coordinates) |
| **Detail Level** | General position | Specific shape/contour/pose |
| **Data Effort** | Low (Box coordinates) | High (Laborious point annotation) |
| **Model Type** | Regression task | Regression task |